# Lab portfolio 1: Geospatial Machine Learning

This is a short practice exercise for developing a machine learning model using `scikit-learn`. The task is to predict leaf area index (LAI) using predictor variables derived from a <a href="https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S2_SR_HARMONIZED" target="_blank">Sentinel-2 satellite image</a> and topographic variables. The LAI measurements were derived from LiDAR data with a 30 cm spatial resolution and 15 cm vertical resolution averaged to a 10 m spatial resolution to match the size of Sentinel-2 pixels.

LAI is a measure of the total area of leaves relative to the ground area and is an important biophysical variable for studying vegetation growth and functioning. The data we are using were collected over the Marburg Forest in Germany and are from the paper by <a href="https://www.sciencedirect.com/science/article/abs/pii/S0304380019303230" target="_blank">Meyer et al. (2019)</a>.

## Setup

### Load data

In [1]:
import os
import subprocess

if "data-geoml" not in os.listdir(os.getcwd()):
    subprocess.run('wget "https://github.com/envt-5566/data/raw/main/data-geoml-2026.zip"', shell=True, capture_output=True, text=True)
    subprocess.run('unzip "data-geoml-2026.zip"', shell=True, capture_output=True, text=True)
    if "data-geoml-2026.zip" not in os.listdir(os.getcwd()):
        print("Has a directory called data-geoml been downloaded and placed in your working directory? If not, try re-executing this code chunk")
    else:
        print("Data download OK")

DATA_PATH = os.path.join(os.getcwd())

Data download OK


### Load packages

In [2]:
if 'google.colab' in str(get_ipython()):
    !pip install mapclassify
    !pip install contextily
    !pip install pysal

import os
import math

import numpy as np
import pandas as pd
import geopandas as gpd

# spatial analysis libraries
import pysal

# plotting
import seaborn as sns
import matplotlib.pyplot as plt

# preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# models
from sklearn.neural_network import MLPRegressor
from sklearn.cluster import KMeans

# metrics
from sklearn.metrics import mean_squared_error

## Load data

In [3]:
gdf = gpd.read_file(os.path.join(DATA_PATH, "data-geoml", "lai_meyer_et_al_2019_marburg.geojson"))
gdf = gdf.drop(columns=["field_1", "ID", "x_utm_25832", "y_utm_25832"])
gdf = gdf.to_crs("EPSG:4326")

In [4]:
gdf.head()

,B02,B03,B04,B08,B05,B06,B07,B11,B12,B8A,dem,slope,aspect,LAI,geometry
0,785,594,372,1852,593.0000,1508.6250,1839.6875,699.6875,290.1250,1960.6875,320.776764,0.245344,5.787806,11.59900,POINT (8.66774 50.8391)
1,778,602,363,1988,597.5625,1575.3125,1939.5625,700.2500,285.6250,2085.8125,322.118378,0.261233,5.744706,10.73155,POINT (8.66788 50.8391)
2,784,611,365,2091,613.6875,1637.9375,2018.1875,717.2500,281.8750,2182.4375,323.610901,0.276209,5.713427,10.15847,POINT (8.66803 50.8391)
3,762,613,361,2038,622.3125,1663.7500,2057.7500,728.0625,284.9375,2231.6875,325.171997,0.295282,5.709146,10.15847,POINT (8.66817 50.8391)
4,780,607,358,2070,623.4375,1652.7500,2058.2500,732.6875,294.8125,2233.5625,326.905609,0.306035,5.740194,9.66879,POINT (8.66831 50.8391)


In [5]:
gdf.explore(column="LAI")

## Activity

Predicting LAI is a machine learning regression task as LAI is a continuous numeric value. **Can you adapt the examples from previous notebooks to develop and evaluate a model that predicts LAI from spectral reflectance and topographic predictors?**

You will need to consider:

* What variables to drop before model training.
* If you need to standardise the training and test data.
* What metric you will use to evaluate the model.
* How you will create training and test splits to evaluate the model (think about the spatial structure of your data).

In [6]:
import warnings
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.model_selection import KFold, LeaveOneGroupOut
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)

target = "LAI"
gdf["aspect_sin"] = np.sin(gdf["aspect"])
gdf["aspect_cos"] = np.cos(gdf["aspect"])
predictors = ["B02","B03","B04","B05","B06","B07","B08","B8A","B11","B12",
              "dem","slope","aspect_sin","aspect_cos"]

X = gdf.loc[:, predictors]
y = gdf.loc[:, target]

# 投影坐标做空间聚类
gdf_proj = gdf.to_crs("EPSG:25832")
coords = np.column_stack((gdf_proj.geometry.x, gdf_proj.geometry.y))
spatial_groups = KMeans(n_clusters=5, random_state=4, n_init=10).fit_predict(coords)


def cross_validate_model(X, y, cv, groups=None):
    """每折训练+测试，返回每折的 RMSE/MAE 和 out-of-fold 预测。"""
    fold_metrics = []
    oof_preds = np.zeros(len(y))

    for fold, (train_idx, test_idx) in enumerate(cv.split(X, y, groups)):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        # 只用本折的训练数据 fit scaler，避免信息泄漏
        scaler = StandardScaler().fit(X_train)

        regr = MLPRegressor(hidden_layer_sizes=(50,), random_state=4,
                            solver="sgd", max_iter=500).fit(scaler.transform(X_train), y_train)

        y_test_preds = regr.predict(scaler.transform(X_test))
        oof_preds[test_idx] = y_test_preds

        fold_metrics.append({
            "fold": fold,
            "n_test": len(test_idx),
            "rmse": np.sqrt(mean_squared_error(y_test, y_test_preds)),
            "mae": mean_absolute_error(y_test, y_test_preds),
        })

    return pd.DataFrame(fold_metrics), oof_preds


random_cv  = KFold(n_splits=5, shuffle=True, random_state=4)
spatial_cv = LeaveOneGroupOut()

random_metrics,  random_oof  = cross_validate_model(X, y, random_cv)
spatial_metrics, spatial_oof = cross_validate_model(X, y, spatial_cv, groups=spatial_groups)

print("Random k-fold, per fold:");  print(random_metrics.round(3))
print("\nSpatial k-fold, per fold:"); print(spatial_metrics.round(3))

comparison = pd.DataFrame({
    "cv_strategy": ["random k-fold", "spatial k-fold"],
    "mean_fold_rmse": [random_metrics["rmse"].mean(), spatial_metrics["rmse"].mean()],
    "oof_rmse": [np.sqrt(mean_squared_error(y, random_oof)),
                 np.sqrt(mean_squared_error(y, spatial_oof))],
    "oof_r2":   [r2_score(y, random_oof), r2_score(y, spatial_oof)],
})
print("\n"); print(comparison.round(3))

Random k-fold, per fold:
   fold  n_test   rmse    mae
0     0     165  1.041  0.655
1     1     165  0.850  0.542
2     2     165  1.189  0.803
3     3     165  1.103  0.723
4     4     164  1.034  0.693

Spatial k-fold, per fold:
   fold  n_test   rmse    mae
0     0     132  0.528  0.438
1     1     107  4.079  3.581
2     2     295  2.463  1.823
3     3     169  1.451  1.163
4     4     121  0.914  0.650


      cv_strategy  mean_fold_rmse  oof_rmse  oof_r2
0   random k-fold           1.043     1.049   0.841
1  spatial k-fold           1.887     2.221   0.289


In [7]:
from scipy.spatial import cKDTree

def nearest_training_distance(coords, cv, X, y, groups=None):
    """每个 hold-out 样本到其训练折中最近样本的距离（米）"""
    dist = np.zeros(len(coords))
    for train_idx, test_idx in cv.split(X, y, groups):
        tree = cKDTree(coords[train_idx])
        dist[test_idx], _ = tree.query(coords[test_idx])
    return dist

gdf["random_train_dist"]  = nearest_training_distance(coords, random_cv, X, y)
gdf["spatial_train_dist"] = nearest_training_distance(coords, spatial_cv, X, y, spatial_groups)
print(gdf[["random_train_dist", "spatial_train_dist"]].describe().round(0))

       random_train_dist  spatial_train_dist
count              824.0               824.0
mean                10.0               330.0
std                  2.0               119.0
min                 10.0                10.0
25%                 10.0               265.0
50%                 10.0               332.0
75%                 10.0               425.0
max                 28.0               556.0


## Activity

Consider the following questions and write about a paragraph in response.

**Outline the rationale behind your strategy for creating training and test splits for model evaluation.**

**Based on your current evaluation strategy, could you be confident in deploying your model to generate accurate LAI predictions for all of Germany? for all of Europe?**

**You are tasked with generating a Germany-wide LAI map, outline a strategy for generating training and test data to support this task and describe why this strategy is suitable.**

Q1:
I created the training and test splits by applying k-means clustering to the point coordinates to form five spatial clusters, and each cluster was held out as the test set in turn. I did not use a random split because the points are densely sampled, so neighbouring points would end up in both the training and test sets; the test set would then not be independent, and the error estimate would be over-optimistic. By holding out whole spatial clusters, the test set covers an entire area, so it tests how well the model predicts where there is no training data. This is supported by my results: R² dropped from 0.841 with random k-fold cross-validation to 0.289 with spatial k-fold cross-validation, and RMSE roughly doubled from 1.049 to 2.221. In addition, held-out points were on average about 10 m from the nearest training point with a random split, compared with about 330 m with a spatial split.

Q2:
No, I would not be confident deploying my model for all of Germany or all of Europe, because the training data only come from one forest in Marburg, about 1.4 km by 1.1 km. Other forest types, farmland, cities, mountains and climates are not represented. Even the spatial cross-validation only tested predictions a few hundred metres away within the same forest. Therefore, the error estimate only applies to conditions similar to the training data, not to the rest of Germany. For Europe, the problem is even greater because there are many more types of vegetation and climate. An area of applicability analysis would likely show that most of Germany and Europe fall outside the conditions supported by the training data.

Q3:
I would use stratified random sampling across Germany. I would divide Germany into strata based on climate zone, land cover or forest type, and elevation, and then randomly place sample points within each stratum, so that the training data cover the full range of predictor values in Germany. For the test data, I would collect a separate probability sample across the whole country, because an ideal test set is a probability sample of the whole target area. I would also use spatial cross-validation to set the dissimilarity index threshold and produce an area of applicability map, which would show where predictions are not supported by the training data. This strategy is suitable because the training data would cover all the main conditions across Germany, the test set would give an accuracy estimate that represents the whole map, and the area of applicability map would show where the predictions can be trusted.